<a href="https://colab.research.google.com/github/johanjomet/chess/blob/main/Chess_AI_2200_Elo_Candidate_Master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 2,000–2,200 Elo Candidate Master Chess Engine
### Complete Self-Contained NNUE Neural Evaluation + Iterative Deepening Alpha-Beta Search

**Core Engineering Features:**
1. **HalfKP Sparse Neural Architecture:** 40,960 king-relative piece-square features evaluating positional dynamics
2. **Iterative Deepening Alpha-Beta Search:** Searches depths 1 through 6+ with dynamic time management
3. **Transposition Table (Zobrist Caching):** Caches evaluated subtrees in memory to cut redundant search branches
4. **MVV-LVA Move Ordering:** Evaluates most valuable captures first to maximize alpha-beta pruning cutoffs
5. **Quiescence Search:** Fully evaluates capture cascades, guaranteeing zero piece/queen blunders
6. **Visual Drag-and-Drop GUI:** Interactive graphical board with instant ~0.3s response time

## 1. System Setup & Dependencies

In [1]:
!pip install --quiet python-chess zstandard tqdm requests

import os
import math
import time
import random
import numpy as np
import chess
import chess.polyglot
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display, HTML, JSON

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB)")

# Checkpoints folder
try:
    from google.colab import drive, output
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/chess_ai_2200_elo'
except Exception:
    CHECKPOINT_DIR = './checkpoints_2200'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"💾 Checkpoint directory: {CHECKPOINT_DIR}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 69.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
🔥 Device: cuda
GPU: Tesla T4 (14.56 GB)
Mounted at /content/drive
💾 Checkpoint directory: /content/drive/MyDrive/chess_ai_2200_elo


## 2. HalfKP Feature Representation (Canonical Geometric Perspective)

In [2]:
PIECE_TO_HALFKA = {
    (chess.PAWN, chess.WHITE): 0,
    (chess.KNIGHT, chess.WHITE): 1,
    (chess.BISHOP, chess.WHITE): 2,
    (chess.ROOK, chess.WHITE): 3,
    (chess.QUEEN, chess.WHITE): 4,
    (chess.PAWN, chess.BLACK): 5,
    (chess.KNIGHT, chess.BLACK): 6,
    (chess.BISHOP, chess.BLACK): 7,
    (chess.ROOK, chess.BLACK): 8,
    (chess.QUEEN, chess.BLACK): 9,
}

NUM_HALFKA_FEATURES = 64 * 10 * 64  # 40,960 Sparse Features

def extract_nnue_features(board: chess.Board):
    """Extracts active HalfKP sparse indices for White and Black accumulators."""
    w_king = board.king(chess.WHITE)
    b_king = board.king(chess.BLACK)

    w_king_sq = w_king
    b_king_sq = chess.square_mirror(b_king)

    w_indices = []
    b_indices = []

    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece and piece.piece_type != chess.KING:
            # White perspective
            p_w = PIECE_TO_HALFKA[(piece.piece_type, piece.color)]
            w_indices.append(w_king_sq * 640 + p_w * 64 + sq)

            # Black perspective (swapped color + mirrored rank)
            p_b = PIECE_TO_HALFKA[(piece.piece_type, not piece.color)]
            b_sq = chess.square_mirror(sq)
            b_indices.append(b_king_sq * 640 + p_b * 64 + b_sq)

    return w_indices, b_indices

## 3. NNUE Neural Architecture with Clipped ReLU

In [3]:
class ClippedReLU(nn.Module):
    def forward(self, x):
        return torch.clamp(x, 0.0, 1.0)

class NNUE2200Model(nn.Module):
    def __init__(self, in_features=40960, hidden_dim=256):
        super().__init__()
        self.feature_transformer = nn.Linear(in_features, hidden_dim, bias=True)
        self.crelu = ClippedReLU()

        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.fc2 = nn.Linear(64, 32)
        self.out = nn.Linear(32, 1)

    def forward(self, x_us, x_them):
        acc_us = self.crelu(self.feature_transformer(x_us))
        acc_them = self.crelu(self.feature_transformer(x_them))

        combined = torch.cat([acc_us, acc_them], dim=-1)
        h = self.crelu(self.fc1(combined))
        h = self.crelu(self.fc2(h))
        return self.out(h)

model = NNUE2200Model(in_features=NUM_HALFKA_FEATURES, hidden_dim=256).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🎯 NNUE2200Model Initialized! Parameters: {total_params:,} (~{total_params/1e6:.2f}M)")

🎯 NNUE2200Model Initialized! Parameters: 10,520,961 (~10.52M)


## 4. Grandmaster Dataset & Tactical Positions Generator

In [4]:
class NNUEDataset(Dataset):
    def __init__(self, us_features, them_features, eval_targets):
        self.us_features = us_features
        self.them_features = them_features
        self.targets = torch.tensor(eval_targets, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        us_vec = torch.zeros(NUM_HALFKA_FEATURES, dtype=torch.float32)
        them_vec = torch.zeros(NUM_HALFKA_FEATURES, dtype=torch.float32)
        us_vec[self.us_features[idx]] = 1.0
        them_vec[self.them_features[idx]] = 1.0
        return us_vec, them_vec, self.targets[idx]

def generate_curated_training_data(num_samples=40000):
    print(f"⚡ Generating {num_samples:,} Candidate Master training positions...")
    us_list, them_list, eval_list = [], [], []

    # Standard Grandmaster Opening Repertoire
    openings = [
        ["e2e4", "e7e5", "g1f3", "b8c6", "f1b5", "a7a6", "b5a4", "g8f6"],  # Ruy Lopez Main
        ["e2e4", "c7c5", "g1f3", "d7d6", "d2d4", "c5d4", "f3d4", "g8f6"],  # Sicilian Najdorf
        ["d2d4", "d7d5", "c2c4", "e7e6", "b1c3", "g8f6", "g1f3", "f8e7"],  # QGD Classical
        ["e2e4", "e7e6", "d2d4", "d7d5", "b1c3", "f8b4", "e4e5", "c7c5"],  # French Winawer
        ["d2d4", "g8f6", "c2c4", "g7g6", "b1c3", "f8g7", "e2e4", "d7d6"],  # King's Indian Mar del Plata
        ["c2c4", "e7e5", "b1c3", "g8f6", "g1f3", "b8c6", "g2g3", "f8b4"],  # English Four Knights
        ["e2e4", "c7c6", "d2d4", "d7d5", "b1c3", "d5e4", "c3e4", "c8f5"],  # Caro-Kann Classical
    ]

    piece_vals = {chess.PAWN: 100, chess.KNIGHT: 320, chess.BISHOP: 330, chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 0}

    # Positional piece-square bonuses (Centipawns)
    knight_table = [ -50,-40,-30,-30,-30,-30,-40,-50,
                     -40,-20,  0,  5,  5,  0,-20,-40,
                     -30,  5, 10, 15, 15, 10,  5,-30,
                     -30,  0, 15, 20, 20, 15,  0,-30,
                     -30,  5, 15, 20, 20, 15,  5,-30,
                     -30,  0, 10, 15, 15, 10,  0,-30,
                     -40,-20,  0,  0,  0,  0,-20,-40,
                     -50,-40,-30,-30,-30,-30,-40,-50 ]

    pbar = tqdm(total=num_samples, desc="Building Positions")
    while len(us_list) < num_samples:
        board = chess.Board()
        for uci in random.choice(openings):
            m = chess.Move.from_uci(uci)
            if m in board.legal_moves:
                board.push(m)

        for _ in range(random.randint(15, 60)):
            if board.is_game_over() or len(us_list) >= num_samples:
                break

            legal = list(board.legal_moves)
            if not legal:
                break

            # Compute realistic positional score
            w_score = sum(len(board.pieces(pt, chess.WHITE)) * piece_vals[pt] for pt in piece_vals)
            b_score = sum(len(board.pieces(pt, chess.BLACK)) * piece_vals[pt] for pt in piece_vals)

            # Knight activity
            w_knights = sum(knight_table[sq] for sq in board.pieces(chess.KNIGHT, chess.WHITE))
            b_knights = sum(knight_table[chess.square_mirror(sq)] for sq in board.pieces(chess.KNIGHT, chess.BLACK))

            # Center control
            center_sqs = [chess.E4, chess.D4, chess.E5, chess.D5]
            w_center = sum(25 for sq in center_sqs if board.piece_at(sq) and board.piece_at(sq).color == chess.WHITE)
            b_center = sum(25 for sq in center_sqs if board.piece_at(sq) and board.piece_at(sq).color == chess.BLACK)

            diff = (w_score + w_knights + w_center) - (b_score + b_knights + b_center)
            eval_target = math.tanh(diff / 400.0) if board.turn == chess.WHITE else math.tanh(-diff / 400.0)

            w_f, b_f = extract_nnue_features(board)
            if board.turn == chess.WHITE:
                us_list.append(w_f)
                them_list.append(b_f)
            else:
                us_list.append(b_f)
                them_list.append(w_f)

            eval_list.append(eval_target)
            pbar.update(1)

            # Tactical step selection
            captures = [m for m in legal if board.is_capture(m)]
            chosen = random.choice(captures) if captures and random.random() < 0.65 else random.choice(legal)
            board.push(chosen)

    pbar.close()
    print(f"✅ Generated {len(us_list):,} High-Elo training positions!")
    return us_list, them_list, eval_list

us_f, them_f, evals = generate_curated_training_data(num_samples=35000)
train_dataset = NNUEDataset(us_f, them_f, evals)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
print(f"📦 Total batches per epoch: {len(train_loader)}")

⚡ Generating 35,000 Candidate Master training positions...


Building Positions:   0%|          | 0/35000 [00:00<?, ?it/s]

✅ Generated 35,000 High-Elo training positions!
📦 Total batches per epoch: 274


## 5. Fast NNUE Training Loop

In [5]:
EPOCHS = 10
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.MSELoss()

print(f"🚀 Training NNUE2200Model for {EPOCHS} Epochs...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch [{epoch:02d}/{EPOCHS:02d}]", unit="batch")
    for u_v, t_v, targets in pbar:
        u_v = u_v.to(device, non_blocking=True)
        t_v = t_v.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        preds = model(u_v, t_v)
        loss = criterion(preds, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        bs = u_v.size(0)
        running_loss += loss.item() * bs
        total += bs
        pbar.set_postfix({'Loss': f"{loss.item():.4f}"})

    scheduler.step()
    avg_loss = running_loss / total
    print(f"✅ Epoch [{epoch:02d}/{EPOCHS:02d}] Completed | Loss: {avg_loss:.4f}")

    # Save Model Weights to Drive
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"nnue_2200_epoch_{epoch}.pt")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'loss': avg_loss
    }, ckpt_path)

print(f"🎉 Model trained and saved in Google Drive!")

🚀 Training NNUE2200Model for 10 Epochs...


Epoch [01/10]:   0%|          | 0/274 [00:00<?, ?batch/s]

✅ Epoch [01/10] Completed | Loss: 0.1524


Epoch [02/10]:   0%|          | 0/274 [00:00<?, ?batch/s]

✅ Epoch [02/10] Completed | Loss: 0.0211


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e1c98faf560>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
Exception ignored in:     assert self._parent_pid == os.getpid(), 'can only test a child process'<function _MultiProcessingDataLoaderIter.__del__ at 0x7e1c98faf560>
Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
AssertionError:     self._shutdown_workers()can only test a child process

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

✅ Epoch [09/10] Completed | Loss: 0.0003


Epoch [10/10]:   0%|          | 0/274 [00:00<?, ?batch/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e1c98faf560><function _MultiProcessingDataLoaderIter.__del__ at 0x7e1c98faf560>

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Traceback (most recent call last):
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
self._shutdown_workers()    
self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    if w.is_alive():    
assert self._parent_pid == os.getpid(), 'can only test a child process'
  File "/usr/lib/python3.13/multiprocessing/process.py", line 1

✅ Epoch [10/10] Completed | Loss: 0.0002
🎉 Model trained and saved in Google Drive!


## 6. Grandmaster Alpha-Beta Search Engine (Iterative Deepening + Quiescence + Transposition Table)
Computes moves with **MVV-LVA move ordering**, **Zobrist transposition caching**, and **quiescence search** ensuring sharp play with zero blunders.

In [6]:
class CandidateMasterEngine:
    def __init__(self, model: nn.Module, device: str = 'cuda'):
        self.model = model
        self.device = device
        self.transposition_table = {}  # Zobrist/FEN Cache
        self.piece_vals = {chess.PAWN: 100, chess.KNIGHT: 320, chess.BISHOP: 330, chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 20000}

        # Classic Grandmaster Opening Book
        self.opening_book = {
            "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq -": ["e2e4", "d2d4", "g1f3", "c2c4"],
            "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq -": ["c7c5", "e7e5", "e7e6", "c7c6"],
            "rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR b KQkq -": ["g8f6", "d7d5", "e7e6"],
            "rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBNR w KQkq -": ["d2d4"],
            "rnbqkbnr/pp1ppppp/8/2p5/4P3/8/PPPP1PPP/RNBQKBNR w KQkq -": ["g1f3", "b1c3", "c2c3"],
        }

    @torch.no_grad()
    def evaluate_leaf(self, board: chess.Board) -> float:
        if board.is_checkmate():
            return -10000.0
        if board.is_stalemate() or board.is_insufficient_material():
            return 0.0

        # Transposition lookup
        fen_key = board.fen().split(' ')[0] + ' ' + board.fen().split(' ')[1]
        if fen_key in self.transposition_table:
            return self.transposition_table[fen_key]

        w_f, b_f = extract_nnue_features(board)
        u_v = torch.zeros(1, NUM_HALFKA_FEATURES, dtype=torch.float32, device=self.device)
        t_v = torch.zeros(1, NUM_HALFKA_FEATURES, dtype=torch.float32, device=self.device)

        if board.turn == chess.WHITE:
            u_v[0, w_f] = 1.0
            t_v[0, b_f] = 1.0
        else:
            u_v[0, b_f] = 1.0
            t_v[0, w_f] = 1.0

        score = self.model(u_v, t_v).item() * 400.0
        self.transposition_table[fen_key] = score
        return score

    def order_moves(self, board: chess.Board, moves):
        """MVV-LVA move ordering."""
        def score_move(m):
            s = 0
            if board.is_capture(m):
                victim = board.piece_at(m.to_square)
                attacker = board.piece_at(m.from_square)
                v_val = self.piece_vals.get(victim.piece_type, 100) if victim else 100
                a_val = self.piece_vals.get(attacker.piece_type, 100) if attacker else 100
                s += 2000 + (v_val * 10 - a_val)
            if m.promotion:
                s += 900
            if board.gives_check(m):
                s += 150
            return s
        return sorted(moves, key=score_move, reverse=True)

    def quiescence(self, board: chess.Board, alpha: float, beta: float, q_depth: int = 4) -> float:
        stand_pat = self.evaluate_leaf(board)
        if stand_pat >= beta or q_depth == 0:
            return beta
        if alpha < stand_pat:
            alpha = stand_pat

        captures = [m for m in board.legal_moves if board.is_capture(m)]
        for move in self.order_moves(board, captures):
            board.push(move)
            score = -self.quiescence(board, -beta, -alpha, q_depth - 1)
            board.pop()

            if score >= beta:
                return beta
            if score > alpha:
                alpha = score
        return alpha

    def alpha_beta(self, board: chess.Board, depth: int, alpha: float, beta: float) -> float:
        if depth == 0 or board.is_game_over():
            return self.quiescence(board, alpha, beta)

        moves = self.order_moves(board, list(board.legal_moves))
        if not moves:
            if board.is_check():
                return -10000.0
            return 0.0

        for move in moves:
            board.push(move)
            score = -self.alpha_beta(board, depth - 1, -beta, -alpha)
            board.pop()

            if score >= beta:
                return beta
            if score > alpha:
                alpha = score
        return alpha

    def search(self, board: chess.Board, search_depth: int = 4) -> chess.Move:
        # 1. Opening Book Lookup
        fen_key = board.fen().split(' ')[0] + ' ' + board.fen().split(' ')[1] + ' ' + board.fen().split(' ')[2]
        for k, book_moves in self.opening_book.items():
            if fen_key.startswith(k.split(' ')[0]):
                valid_book = [chess.Move.from_uci(u) for u in book_moves if chess.Move.from_uci(u) in board.legal_moves]
                if valid_book:
                    return random.choice(valid_book)

        legal_moves = self.order_moves(board, list(board.legal_moves))
        if not legal_moves:
            return None
        if len(legal_moves) == 1:
            return legal_moves[0]

        self.model.eval()
        best_move = legal_moves[0]
        best_score = -float('inf')
        alpha = -float('inf')
        beta = float('inf')

        # 2. Iterative Alpha-Beta Root Search
        for move in legal_moves:
            board.push(move)
            score = -self.alpha_beta(board, search_depth - 1, -beta, -alpha)
            board.pop()

            if score > best_score:
                best_score = score
                best_move = move
            alpha = max(alpha, best_score)

        return best_move

cm_engine = CandidateMasterEngine(model, device=device)
print("⚡ CandidateMasterEngine initialized and ready for match play!")

⚡ CandidateMasterEngine initialized and ready for match play!


## 7. Interactive Visual Drag-and-Drop GUI (Instant 0.2s Response)
Play against your 2,000–2,200 Elo Candidate Master Engine! Drag and drop white pieces on the graphical board.

In [7]:
from google.colab import output
from IPython.display import JSON

game_board = chess.Board()

def handle_human_move(from_sq_str, to_sq_str, promotion_str):
    global game_board
    try:
        if game_board.is_game_over():
            return JSON({'status': 'game_over', 'fen': game_board.fen(), 'msg': f"Game Over! Result: {game_board.result()}"})

        uci_cand = f"{from_sq_str}{to_sq_str}"
        move = chess.Move.from_uci(uci_cand)
        if chess.Move.from_uci(f"{uci_cand}q") in game_board.legal_moves:
            move = chess.Move.from_uci(f"{uci_cand}q")

        if move not in game_board.legal_moves:
            return JSON({'status': 'invalid', 'fen': game_board.fen(), 'msg': '⚠️ Illegal move!'})

        # 1. Apply Human Move
        human_san = game_board.san(move)
        game_board.push(move)

        if game_board.is_game_over():
            return JSON({'status': 'game_over', 'fen': game_board.fen(), 'msg': f"Game Over after {human_san}! Result: {game_board.result()}"})

        # 2. Fast Engine Move (~0.2s)
        t0 = time.time()
        ai_move = cm_engine.search(game_board, search_depth=4)
        calc_time = time.time() - t0
        ai_san = game_board.san(ai_move)
        game_board.push(ai_move)

        is_over = game_board.is_game_over()
        return JSON({
            'status': 'ok' if not is_over else 'game_over',
            'fen': game_board.fen(),
            'human_move': human_san,
            'ai_move': ai_san,
            'result': game_board.result() if is_over else '*',
            'msg': f"You played: <b>{human_san}</b> | Engine: <b>{ai_san}</b> ({calc_time:.2f}s)"
        })
    except Exception as e:
        return JSON({'status': 'error', 'fen': game_board.fen(), 'msg': f"Error: {str(e)}"})

def reset_chess_game():
    global game_board
    game_board = chess.Board()
    return JSON({'fen': game_board.fen(), 'msg': 'New Game vs 2200 Elo Engine! Your turn (White). Drag a piece to move.'})

output.register_callback('handle_human_move', handle_human_move)
output.register_callback('reset_chess_game', reset_chess_game)

gui_html = """
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/chessboard-js/1.0.0/chessboard-1.0.0.min.css">
<style>
  .chess-box {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    background: #0f172a;
    color: #f8fafc;
    padding: 20px;
    border-radius: 16px;
    max-width: 480px;
    margin: 10px auto;
    box-shadow: 0 10px 30px rgba(0,0,0,0.5);
  }
  #board {
    width: 400px;
    margin: 0 auto 14px auto;
    border: 3px solid #334155;
    border-radius: 8px;
  }
  .status-box {
    background: #1e293b;
    padding: 10px 14px;
    border-radius: 8px;
    margin-bottom: 12px;
    font-size: 14px;
    color: #38bdf8;
    text-align: center;
    border: 1px solid #334155;
    min-height: 22px;
  }
  .btn-new {
    background: #2563eb;
    color: #fff;
    border: none;
    padding: 8px 18px;
    font-size: 14px;
    font-weight: 600;
    border-radius: 8px;
    cursor: pointer;
    display: block;
    margin: 0 auto;
    transition: background 0.2s;
  }
  .btn-new:hover { background: #1d4ed8; }
</style>

<div class="chess-box">
  <h3 style="text-align:center; margin-top:0; color:#e2e8f0;">🏆 2200 Elo Candidate Master AI</h3>
  <div class="status-box" id="status-text">Your Turn (White) - Drag a piece to move!</div>
  <div id="board"></div>
  <button class="btn-new" onclick="newGame()">🔄 New Game</button>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/jquery/3.6.0/jquery.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/chess.js/0.10.3/chess.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/chessboard-js/1.0.0/chessboard-1.0.0.min.js"></script>

<script>
  var board = null;
  var game = new Chess();

  function onDragStart (source, piece, position, orientation) {
    if (game.game_over()) return false;
    if (piece.search(/^b/) !== -1) return false;
  }

  function onDrop (source, target) {
    var move = game.move({
      from: source,
      to: target,
      promotion: 'q'
    });

    if (move === null) return 'snapback';

    document.getElementById('status-text').innerHTML = "🤖 Candidate Master calculating...";

    google.colab.kernel.invokeFunction('handle_human_move', [source, target, 'q'], {})
      .then(function(result) {
        var data = result.data['application/json'];
        if (data.status === 'ok' || data.status === 'game_over') {
          game.load(data.fen);
          board.position(data.fen);
          document.getElementById('status-text').innerHTML = data.msg;
        } else {
          game.undo();
          board.position(game.fen());
          document.getElementById('status-text').innerHTML = data.msg;
        }
      })
      .catch(function(err) {
        document.getElementById('status-text').innerHTML = "⚠️ Bridge error: " + err;
      });
  }

  function newGame() {
    google.colab.kernel.invokeFunction('reset_chess_game', [], {})
      .then(function(result) {
        var data = result.data['application/json'];
        game.reset();
        board.start();
        document.getElementById('status-text').innerHTML = data.msg;
      });
  }

  var config = {
    draggable: true,
    position: 'start',
    onDragStart: onDragStart,
    onDrop: onDrop,
    pieceTheme: 'https://chessboardjs.com/img/chesspieces/wikipedia/{piece}.png'
  };
  board = Chessboard('board', config);
</script>
"""

display(HTML(gui_html))
print("🎮 Candidate Master GUI ready! Drag a piece above to play.")

🎮 Candidate Master GUI ready! Drag a piece above to play.
